In [ ]:
import os
import sys

# sys.path.append(os.path.expanduser("~/repos/cax-scripts"))
# sys.path.append(os.path.expanduser("~/scripts"))
sys.path.append(os.path.expanduser("~/repos/cax-control"))
import matplotlib.pyplot as plt

# from Shadow import histo
import numpy as np
from shadowpy.si_beamlines import CAXSim
from shadowpy.utils import save_image, Image

from image_statistics import Histogram2DAnalyzer
from caustic import CausticResult, image_caustic


In [ ]:
cax = CAXSim(total_rays=100_000, image_size=(210, 280))


In [ ]:
def ana_from_image(image: Image):

    ana = Histogram2DAnalyzer(
        img=image.histogram,
        x_bin_edges=image.bin_h_edges*1000,
        y_bin_edges=image.bin_v_edges*1000
    )

    return ana

In [ ]:
ana = ana_from_image(cax.dvf_B1.image)

In [ ]:
cax.mirror.offset = [0.3, -0.01, 0.0]
cax.mirror.rx = -0.0025
cax.mirror.ry = 0.005

In [ ]:
ana.fit(ana.compute_momenta())

In [ ]:
plt.imshow(ana.img)

In [ ]:
fig, ax = ana.plot(hprm=ana.hprm_fitting)

# Rasterize the pcolormesh to reduce the size of the PDF file
for c in ax.collections:
    c.set_rasterized(True)

ax.set_title("Beam Fitted Model")
ax.set_xlabel("x [μm]")
ax.set_ylabel("y [μm]")

In [ ]:
fig.savefig("/home/gabriel.amici/Documents/CEC/beam_anatomy.pdf", dpi=300, bbox_inches="tight")
fig.savefig("/home/gabriel.amici/Documents/CEC/beam_anatomy.png", dpi=300, bbox_inches="tight")

In [ ]:
def intensity(ana: Histogram2DAnalyzer):

    X = ana.x_bin_centers
    Y = ana.y_bin_centers

    center = np.array([ana.hprm_fitting['mux'], ana.hprm_fitting['muy']])
    theta = ana.hprm_fitting['theta']

    sig1 = ana.hprm_fitting['sig_major']
    sig2 = ana.hprm_fitting['sig_minor']

    xp = (X-center[0])*np.cos(-theta) + \
        (Y[:, None]-center[1])*np.sin(-theta)

    yp = -(X-center[0])*np.sin(-theta) + \
        (Y[:, None]-center[1])*np.cos(-theta)

    ellipse = (xp/(2*sig1))**2 + (yp/(2*sig2))**2 <= 1
    beam_region = np.where(ellipse, ana.img, 0)

    return np.sum(beam_region)

In [ ]:
cax = CAXSim(image_size=(100, 100))

In [ ]:
%timeit cax.mirror.tx = np.random.rand() ; ana = ana_from_image(cax.dvf_B1.image) ; ana.fit(hprm=ana.compute_momenta())

In [ ]:
caustic_res = image_caustic(cax.dvf_B1.beam, 
                            cax.dvf_B1.frame, 
                            np.linspace(-100, 100, 30),
                            image_size=(100, 100),)

In [ ]:
ana.print_stats(hprm=ana.hprm_fitting)

In [ ]:
ana.plot(hprm=ana.hprm_fitting, roi_zoom=None)

In [ ]:
intensity(ana), ana.hprm_fitting['amplitude']

In [ ]:
ana.hprm_fitting['amplitude']

In [ ]:
beam1 = cax.dvf_B1.beam.duplicate()
beam1.retrace(-118.98)

In [ ]:
print(beam1)

In [ ]:
img = save_image(cax.dvf_B1, beam1, image_size=(100, 100))


In [ ]:
ana = ana_from_image(img)
ana.fit(hprm=ana.compute_momenta())

In [ ]:
cax.mirror.tx = 4

In [ ]:
ana.print_stats(ana.hprm_fitting)

In [ ]:
ana.plot(hprm=ana.hprm_fitting)

In [ ]:
11.5/19.9

In [ ]:
cax.mirror.tx = 0 ; cax.mirror.ry = 0

In [ ]:
cax = CAXSim(image_size=(100, 100))

In [ ]:
amps_tx = []
amps_err = []
# flux_tx = []
txs  = np.linspace(-2, 2, 31)
for _tx in txs:
    cax.mirror.tx = _tx
    ana = ana_from_image(cax.dvf_B1.image)
    ana.compute_momenta()
    ana.fit(hprm=ana.hprm_momenta)

    amps_tx.append(ana.hprm_fitting['amplitude'] if ana.beam_visible else np.nan)
    amps_err.append(ana.hprm_fitting['errors']['amplitude'] if ana.beam_visible else np.nan)
    # flux_tx.append(intensity(ana)/cax.dvf_B1.image.bin_area if ana.beam_visible else np.nan)


# [print(f"Peak = {amp:+.2f}") for amp in amps]
    

In [ ]:
sigx_tx = []
sigx_err = []
# flux_tx = []
txs  = np.linspace(-2, 2, 31)
for _tx in txs:
    cax.mirror.tx = _tx
    ana = ana_from_image(cax.dvf_B1.image)
    ana.compute_momenta()
    ana.fit(hprm=ana.hprm_momenta)

    sigx_tx.append(ana.hprm_fitting['sigx'] if ana.beam_visible else np.nan)
    sigx_err.append(ana.hprm_fitting['errors']['sigx'] if ana.beam_visible else np.nan)
    # flux_tx.append(intensity(ana)/cax.dvf_B1.image.bin_area if ana.beam_visible else np.nan)


# [print(f"Peak = {amp:+.2f}") for amp in amps]
    

In [ ]:
ana.plot(hprm=ana.hprm_momenta)

In [ ]:
ana.print_stats(hprm=ana.hprm_fitting)

In [ ]:
cax.mirror.tx = 0

In [ ]:
amps_ry = []
# flux_ry = []
rys  = np.linspace(-0.2, 0.2, 31)
for _ry in rys:
    cax.mirror.ry = _ry
    ana = ana_from_image(cax.dvf_B1.image)
    ana.compute_momenta()
    ana.fit(hprm=ana.hprm_momenta)

    amps_ry.append(ana.hprm_fitting['amplitude'])
    # flux_ry.append(intensity(ana))


# [print(f"Peak = {amp:+.2f} μm") for amp in amps_ry]
    

In [ ]:
fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('TX [mm]')
# ax1.set_ylabel('exp', color=color)
ax1.errorbar(txs, amps_tx, yerr=amps_err, color=color, label='Amplitude (fitting)', marker='o')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(color=color)

# ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

# color = 'tab:blue'
# # ax2.set_ylabel('sin', color=color)  # we already handled the x-label with ax1
# ax2.plot(txs, flux_tx, color=color, label='Flux', marker='o')
# ax2.tick_params(axis='y', labelcolor=color)
fig.legend()
# ax2.grid(color=color)
fig.tight_layout()  # otherwise the right y-label is slightly clipped
# plt.ylim(0, 60)
plt.show()

In [ ]:
fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('TX [mm]')
# ax1.set_ylabel('exp', color=color)
ax1.errorbar(txs, sigx_tx, yerr=sigx_err, color=color, label='Sigma X (fitting)', marker='o')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(color=color)

# ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

# color = 'tab:blue'
# # ax2.set_ylabel('sin', color=color)  # we already handled the x-label with ax1
# ax2.plot(txs, flux_tx, color=color, label='Flux', marker='o')
# ax2.tick_params(axis='y', labelcolor=color)
fig.legend()
# ax2.grid(color=color)
fig.tight_layout()  # otherwise the right y-label is slightly clipped
# plt.ylim(0, 60)
plt.show()

In [ ]:
amps_err

In [ ]:
amps_err

In [ ]:
fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('TX [mm]')
# ax1.set_ylabel('exp', color=color)
ax1.errorbar(txs, amps_tx, yerr=amps_err, color=color, label='Amplitude (fitting)', marker='o')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(color=color)

# ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

# color = 'tab:blue'
# # ax2.set_ylabel('sin', color=color)  # we already handled the x-label with ax1
# ax2.plot(txs, flux_tx, color=color, label='Flux', marker='o')
# ax2.tick_params(axis='y', labelcolor=color)
fig.legend()
# ax2.grid(color=color)
fig.tight_layout()  # otherwise the right y-label is slightly clipped
plt.xlim(-1, 1)
plt.show()

In [ ]:
txs[np.nanargmax(amps_tx)], amps_tx[np.nanargmax(amps_tx)]

In [ ]:
cax.mirror.tx = -2.9333333333333336

ana = ana_from_image(cax.dvf_B1.image)
ana.plot(hprm=ana.hprm_fitting)

In [ ]:
fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('time (s)')
# ax1.set_ylabel('exp', color=color)
ax1.plot(rys, amps_ry, color=color, label='Amplitude (fitting)')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

color = 'tab:blue'
# ax2.set_ylabel('sin', color=color)  # we already handled the x-label with ax1
ax2.plot(rys, flux_ry, color=color, label='Flux')
ax2.tick_params(axis='y', labelcolor=color)
fig.legend()
fig.tight_layout()  # otherwise the right y-label is slightly clipped
plt.show()

In [ ]:
plt.plot(rys, amps_ry, marker='o', label="Amplitude (fit)")
plt.plot(rys, flux_ry, marker='o', label="Peak (droi)")
plt.xlabel('RY [mrad]')
plt.ylabel('Peak [counts]')
plt.grid()
plt.legend()


In [ ]:
q_foci = -118.98

In [ ]:
amps_ry = []
flux_ry = []
rys  = np.linspace(-0.2, 0.2, 31)
for _ry in rys:
    cax.mirror.ry = _ry
    cax.trace()
    beamr = cax.dvf_B1.beam.duplicate()
    beamr.retrace(q_foci)
    img = save_image(cax.dvf_B1, beamr, image_size=(100, 100))
    ana = ana_from_image(img)
    ana.compute_momenta()
    ana.fit(hprm=ana.hprm_momenta)

    amps_ry.append(ana.hprm_fitting['amplitude']/cax.dvf_B1.image.bin_area if ana.beam_visible else np.nan)
    flux_ry.append(intensity(ana)/cax.dvf_B1.image.bin_area if ana.beam_visible else np.nan)


# [print(f"Peak = {amp:+.2f} μm") for amp in amps_ry]
    

In [ ]:
fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('RY [μrad]')
# ax1.set_ylabel('exp', color=color)
ax1.plot(rys*1000, amps_ry, color=color, label='Amplitude (fitting)', marker='o')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(color=color)

# ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

# color = 'tab:blue'
# # ax2.set_ylabel('sin', color=color)  # we already handled the x-label with ax1
# ax2.plot(rys*1000, flux_ry, color=color, label='Flux', marker='o')
# ax2.tick_params(axis='y', labelcolor=color)
fig.legend()
# ax2.grid(color=color)
fig.tight_layout()  # otherwise the right y-label is slightly clipped
plt.show()

In [ ]:
cax.mirror.ry = 0
ana = ana_from_image(cax.dvf_B1.image)
ana.fit()
ana.plot(hprm=ana.hprm_fitting)
print(ana.hprm_fitting['amplitude'])
print(ana.hprm_project['amplitude'])

In [ ]:
cax.mirror.ry = 0.01
ana = ana_from_image(cax.dvf_B1.image)
ana.fit()
ana.plot(hprm=ana.hprm_fitting)
print(ana.hprm_fitting['amplitude'])
print(ana.hprm_project['amplitude'])

In [ ]:
from optlnls.

In [ ]:
cax.mask_A1.shadow_oe.RX_SLIT[0] = 10.0
cax.mask_A1.shadow_oe.RZ_SLIT[0] = 10.0



In [ ]:
print(f"ΔTX = {np.ptp(rys):.2f} mm")
print(f"ΔCx = {np.ptp(amps)*0.48:.2f} μm")

In [ ]:
cax.mirror.offset# = np.random.normal(0, 1/2, 3)
cax.mirror.tilt = np.random.normal(0, 0.0001, 3)
cax.mirror.tilt

In [ ]:
ana_source = cax.source.image

fig, ax = plt.subplots(figsize=(10, 6))
fig, ax = ana_source.plot( fig=fig, ax=ax )
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
ax.set_title("Image at source ($z = 0$ m)")

In [ ]:
cax.mirror.ry = 0.1

In [ ]:
cax.dvf_B1.image.plot()

In [ ]:
# Propagated
source_beam = cax.beams[0].duplicate()
source_beam.retrace(17000) #17 m
ana_at_mirror = save_image(cax.source, source_beam)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
m = ax.pcolormesh(
            ana_at_mirror.x_bin_edges, ana_at_mirror.y_bin_edges, ana_at_mirror.img.T,
            shading="auto", cmap="viridis",
        )
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
plt.colorbar(m, ax=ax, label="count")
ax.set_title("Image before mirror ($z = 17^-$ m)")
plt.show()

In [ ]:
image_ticket.keys()

In [ ]:
mirror_beam = cax.mirror.beam.duplicate()

image_ticket = mirror_beam.histo2(col_h=2, col_v=1, 
                                   nbins_h=280, nbins_v=210, 
                                   nolost=1)

histogram   = image_ticket["histogram"]
bin_h_centers = image_ticket["bin_h_center"]
bin_v_centers = image_ticket["bin_v_center"]

fig, ax = plt.subplots(figsize=(7, 6))
m = ax.pcolormesh(
            bin_h_centers, bin_v_centers, histogram.T,
            shading="auto", cmap="viridis",
        )
ax.set_xlabel("z [mm]")
ax.set_ylabel("y [mm]")
plt.colorbar(m, ax=ax, label="count")
ax.set_title("Beam along the mirror surface")
plt.show()

In [ ]:
# Propagated
mirror_beam = cax.mirror.beam.duplicate()
# source_beam.retrace(17000) #17 m
ana_after_mirror = save_image(cax.mirror, mirror_beam)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
m = ax.pcolormesh(
            ana_after_mirror.x_bin_edges, ana_after_mirror.y_bin_edges, ana_after_mirror.img.T,
            shading="auto", cmap="viridis",
        )
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
plt.colorbar(m, ax=ax, label="count")
ax.set_title("Image after mirror ($z = 17^+$ m)")
plt.show()

In [ ]:
cax.dvf_A1.source_distance

In [ ]:
# Propagated
slit_beam = cax.mirror.beam.duplicate()
slit_beam.retrace(4359)
# source_beam.retrace(17000) #17 m
ana_before_slit = save_image(cax.dvf_A1, slit_beam)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
m = ax.pcolormesh(
            ana_before_slit.x_bin_edges, ana_before_slit.y_bin_edges, ana_before_slit.img.T,
            shading="auto", cmap="viridis",
        )
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
plt.colorbar(m, ax=ax, label="count")
ax.set_title("Image before slit ($z = 21.359$ m)")
plt.show()

In [ ]:
cax.mirror.ry = -0.1

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig, ax = cax.dvf_B1.image.plot(fig=fig, ax=ax)
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
ax.set_title("Image at screen ($z = 34$ m)")

In [ ]:
cx = cax.dvf_B1.image.hprm_project['mux']
cy = cax.dvf_B1.image.hprm_project['muy']
# cx, cy = cy, cx

fx = cax.dvf_B1.image.hprm_project['fwhmx']
fy = cax.dvf_B1.image.hprm_project['fwhmy']

cax.dvf_B1.image.img[cx, cy]
print(cx, cy)
np.unravel_index(cax.dvf_B1.image.img.argmax(), cax.dvf_B1.image.img.shape)


In [ ]:
cax.dvf_B1.image.x_bin_centers[cx]

In [ ]:
cax.dvf_B1.image.qck_intensity(cax.dvf_B1.image.img, (cx, cy), (fx, fy), 1)

In [ ]:
image_ticket = cax.dvf_B1.beam.histo2(col_h=3, col_v=1, 
                            nbins_h=280, nbins_v=210, nolost=1)

In [ ]:
hist = np.array(image_ticket['histogram'])
h_centers = np.array(image_ticket['bin_h_center'])
v_centers = np.array(image_ticket['bin_v_center'])

In [ ]:
cax.dvf_B1.image.plot(hprm=cax.dvf_B1.image.hprm_momenta)

In [ ]:
from optlnls.plot import plot_beam


In [ ]:
x = h_centers
y = v_centers
img = hist

beam2D_2 = np.zeros((len(y)+1, len(x)+1))
beam2D_2[0,1:] = x
beam2D_2[1:,0] = y
beam2D_2[1:,1:] = img.T


plot_beam(beam2D_2, cut=0, textA=1, textB=10, textC=9, textD=0, fitType=1,
          plot_title='CAX M1 nominal', show_colorbar=1,
            x_range=1, y_range=1, x_range_min=-0.12, x_range_max=0.12, y_range_min=-0.12, y_range_max=0.12,
          zero_pad_x=1, zero_pad_y=1, cmap='viridis', integral=1e11,
          units=2)


In [ ]:
x = cax.dvf_B1.image.x_bin_centers
y = cax.dvf_B1.image.y_bin_centers
img = cax.dvf_B1.image.img

beam2D = np.zeros((len(y)+1, len(x)+1))
beam2D[0,1:] = x
beam2D[1:,0] = y
beam2D[1:,1:] = img.T


plot_beam(beam2D, cut=0, textA=1, textB=10, textC=3, textD=0, fitType=1,
          plot_title='CAX M1 nominal', show_colorbar=1,
            x_range=1, y_range=1, x_range_min=-0.12, x_range_max=0.12, y_range_min=-0.12, y_range_max=0.12,
          zero_pad_x=1, zero_pad_y=1, cmap='viridis', integral=1e11,
          units=2)


In [ ]:
print(cax.dvf_B1.image.hprm_fitting['mux']*1000)
print(cax.dvf_B1.image.hprm_fitting['muy']*1000)

In [ ]:
cax.dvf_B1.image.print_stats(cax.dvf_B1.image.hprm_fitting)

In [ ]:
cax.dvf_B1.image.img.shape